# Market Basket Items Association with Apriori

In [ ]:
import pandas as pd
from apyori import apriori

## Dataset

### Import and Conversion

In [ ]:
market_baskets_df = pd.read_csv("datasets/market-basket.csv", header=None)
display(market_baskets_df)

In [ ]:
market_baskets_list:list[list[str]] = [
    [item for item in sublist if pd.notna(item)]
    for sublist in market_baskets_df.values.tolist()
]
print(market_baskets_list)

## Apply Apriori

### Fit

In [ ]:
relation_records = list(apriori(
    transactions=market_baskets_list,
    min_support=0.005,
    min_confidence=0.05,
    min_lift=2.0
))

print(relation_records.__len__())

### Format

In [ ]:
def format_rules_dataframe(relation_records:list) -> pd.DataFrame:
    rules:list[dict] = []
    for relation in relation_records:
        for statistic in relation.ordered_statistics:
            rules.append({
                "IF": list(statistic.items_base),
                "THEN": list(statistic.items_add),
                "LIFT": statistic.lift,
                "CONFIDENCE": statistic.confidence,
            })

    return pd.DataFrame(rules).sort_values(
        by=["LIFT", "CONFIDENCE"], 
        ascending=[False, False]
    ).reset_index(drop=True)

rules_df = format_rules_dataframe(relation_records=relation_records)
display(rules_df)

rules_df.to_csv("datasets/rules.csv", sep=";", index=False)